# Teleportation

In [9]:
from qiskit import *
from qiskit_aer import *
from qiskit.quantum_info import *
from qiskit.visualization import *
from qiskit.circuit.library import StatePreparation
import numpy as np

## plot function

In [3]:
def qplot(x):
    return x.draw(fold=-1, scale=0.8, cregbundle=False)

In [17]:
# creating the quantum circuit
qa = QuantumRegister(2, name = 'alice')
qb = QuantumRegister(1, name = 'bob')
cbits = ClassicalRegister(2, name = 'cbits')

qc = QuantumCircuit(qa, qb, cbits)

qc.h(qa[1])
qc.cx(qa[1], qb)
qc.barrier(label='Bell state')

# Initialization
ampl = [25, 10]
ampl_norm = ampl/np.linalg.norm(ampl)

qc.append(StatePreparation(ampl_norm), [qa[0]])


# Alice operations
qc.cx(qa[0], qa[1])
qc.h(qa[0])
qc.barrier()

for i in range(2):
    qc.measure(qa[i], cbits[i])


# After sendign the measurement result to Bob, Bob will perform the following conditional operations
qc.z(qb).c_if(cbits[0],1)
qc.x(qb).c_if(cbits[1],1)



qplot(qc)

Bell state ┌────────────────────────────────────┐     ┌───┐ ░ ┌─┐             
alice_0: ───────────────░──────┤ State Preparation(0.92848,0.37139) ├──■──┤ H ├─░─┤M├─────────────
         ┌───┐          ░      └────────────────────────────────────┘┌─┴─┐└───┘ ░ └╥┘┌─┐          
alice_1: ┤ H ├──■───────░────────────────────────────────────────────┤ X ├──────░──╫─┤M├──────────
         └───┘┌─┴─┐     ░                                            └───┘      ░  ║ └╥┘┌───┐┌───┐
    bob: ─────┤ X ├─────░───────────────────────────────────────────────────────░──╫──╫─┤ Z ├┤ X ├
              └───┘     ░                                                       ░  ║  ║ └─╥─┘└─╥─┘
cbits_0: ══════════════════════════════════════════════════════════════════════════╩══╬═══■════╬══
                                                                                      ║        ║  
cbits_1: ═════════════════════════════════════════════════════════════════════════════╩════════■══

In [18]:
ampl_norm

array([0.92847669, 0.37139068])

In [20]:
backend = StatevectorSimulator()

In [27]:
transpilde_qc = transpile(qc, backend)
job = backend.run(transpilde_qc)
result = job.result()
sv = result.get_statevector()

In [32]:
sv.draw('latex')

<IPython.core.display.Latex object>

$$(0.92847669 |0\rangle + 0.37139068 |1\rangle)_{Bob} \otimes |01\rangle_{Alice} $$